# Template - Resumable Colab experiment

Copy this notebook for every long-running experiment. Built around `ResumableRun`
from `quest_kg.utils.checkpoint`: checkpoints to Drive every 50 items, resumes
across session disconnects, atomic writes.

Run `00_setup.ipynb` first.

In [ ]:
import os
os.chdir('/content/quest-kg-cikm2026')

EXPERIMENT = 'EXPERIMENT_NAME'           # e.g. 'headline_webqsp_llama32_seed0'
CKPT  = f'/content/drive/MyDrive/quest_kg/ckpts/{EXPERIMENT}.pkl'
OUT   = f'/content/drive/MyDrive/quest_kg/results/{EXPERIMENT}.csv'
LOG   = f'/content/drive/MyDrive/quest_kg/logs/{EXPERIMENT}.log'
print('ckpt:', CKPT)

In [ ]:
# Load your inputs
queries = [...]   # list of items to process

# Define one-item processing function
def process(item):
    # ... call QUEST-KG pipeline on item ...
    return {'qid': item['qid'], 'answer': '...', 'confidence': 0.95}

In [ ]:
from quest_kg.utils.checkpoint import ResumableRun
from tqdm import tqdm

with ResumableRun(CKPT, total=len(queries), every=50) as run:
    for i, q in tqdm(run.iter(queries), total=len(queries), initial=run.state.next_idx):
        r = process(q)
        run.append(r)

# Save final CSV
import pandas as pd
pd.DataFrame(run.results).to_csv(OUT, index=False)
print('wrote', OUT)